# processed_regional — 지자체 문화·관광·문화재 세출 통합 전처리

**베이스(뼈대): 진경 노트북(`regional_expenditure_jinkyeong.ipynb`)**
검증 로직(결측/이상치/손상레코드/공식통계 대조 성격의 회계항등식 검증)을 그대로 채택.

**표면: 승희 노트북(`eda_regional_expenditure.ipynb`)**
컬럼명 영문화 규칙과, 팀 방법론 문서(트랙별 분석방법론 정리)에 기재된 회계유형 코드 체계(1=일반/2=특별/3=기금/4=기타)를 채택.

## 채택 근거 요약 (대화 검증 결과)

| 항목 | 채택 | 근거 |
|---|---|---|
| 정규화 → 중복제거 순서 | 진경 | 공백 정규화를 먼저 해야 인천 옹진군 "홍보책자 제작" 위장 중복(공백 1칸 차이, 960만원)을 잡을 수 있음. 실측: 진경 방식 48,619→48,617행(2건), 승희 방식 48,619→48,618행(1건만 제거) |
| 손상레코드 검사 6종 | 진경 | 회계항등식, 잔액식, 완전중복, HTML엔티티, 공백, 동일키중복을 전수로 분리해 표로 정리 |
| 이상치(금액) | 동일(IQR 미적용) | 세출은 정상적으로 몇백만~몇백억까지 편차가 커서 IQR 적용 시 정상 대형사업을 오분류함. 초과집행(10건)·0원행(524건)만 플래그로 관리 |
| local_own_revenue 정의 | 동일 | 시도비+시군구비 (국비 제외) — 가설2 핵심 지표, 방법론 확정사항과 일치 |
| 회계유형 코드 | **승희(1=일반/2=특별/3=기금/4=기타)** | 방법론 문서·팀 최종 결정사항에 기재된 코드 체계. 분류 로직(단어 매칭 우선순위)은 진경 것을 그대로 사용해 안전성 확보 |
| 컬럼명 | 승희(영문 snake_case) | 팀 네이밍 규칙 |
| 시도×연도 집계본 | 저장하지 않음(참고용 함수만) | 진경 방침 — 집계 조건(회계유형·초과집행 포함 여부)이 바뀌면 파일이 어긋나므로, 필요 시 이 노트북의 함수를 그대로 재사용 |

## 이번 통합에서 추가한 것
- `region_type` (광역시/도) 컬럼 — 트랙1 4분면 분석에서 광역시·도를 층화 비교할 때 바로 쓸 수 있도록 추가. (국민여행조사 유저가이드 141쪽 공식 정의: 광역시는 구간 이동이 여행으로 안 잡히고, 도는 시/군간 이동이 여행으로 잡히므로 방문건수 척도가 서로 다름 — 이 정보를 세출 쪽에도 동일하게 붙여, 국민여행조사와 조인한 뒤 그룹별 분석을 바로 할 수 있게 함)

## 처리 순서
0 준비 → 1 결측치 → 2 이상치 → 3 손상 레코드(정규화→중복제거) → 4 파생변수 → 5 영문 컬럼명 변경 → 6 저장(`processed_regional.csv`)


---
## 0. 준비

In [1]:
import os
import re
import html

import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

SRC_PATH = 'regional_expenditure_2023_2024_2025.csv'
OUT_PATH = 'processed_regional.csv'

AMOUNT_COLS = ['계', '국비', '시도비', '시군구비', '기타', '지출액', '집행잔액']
TEXT_COLS = ['지역', '자치단체', '회계', '세부사업명', '분야', '세부분야']

# 금액 컬럼은 원본 문자열을 그대로 읽은 뒤 형식을 직접 검증한다
raw = pd.read_csv(SRC_PATH, dtype=str, keep_default_na=False)
print(f'원본: {raw.shape[0]:,}행 x {raw.shape[1]}컬럼')

for col in AMOUNT_COLS:
    invalid = raw[~raw[col].str.match(r'^-?\d+(\.\d+)?$')]
    assert len(invalid) == 0, f'{col}: 비수치 형식 {len(invalid)}건 발견'
print('금액 컬럼 7개 전부 정상 숫자 형식 확인')

df = raw.copy()
for col in AMOUNT_COLS:
    df[col] = pd.to_numeric(df[col])
df['연도'] = df['연도'].astype(int)

step_log = [('원본', df.shape[0], df.shape[1])]
print(df.dtypes.to_string())
print()
print('연도별 행수:', df.groupby('연도').size().to_dict())
print('지역:', df['지역'].nunique(), '개')
print('세부분야:', df['세부분야'].value_counts().to_dict())

원본: 48,619행 x 14컬럼
금액 컬럼 7개 전부 정상 숫자 형식 확인


지역         str
자치단체       str
회계         str
세부사업명      str
계        int64
국비       int64
시도비      int64
시군구비     int64
기타       int64
지출액      int64
집행잔액     int64
분야         str
세부분야       str
연도       int64

연도별 행수: {2023: 16192, 2024: 16066, 2025: 16361}
지역: 17 개
세부분야: {'관광': 22419, '문화재': 21589, '문화및관광일반': 4611}


---
## 1. 결측치

NaN / 빈 문자열 / 공백을 모두 결측 후보로 본다 (행정 집계표 특성상 '모름' 같은 코드는 없음).

In [2]:
rows = []
for col in raw.columns:
    nan_count = raw[col].isna().sum()
    blank_count = (raw[col].str.strip() == '').sum()
    rows.append({'column': col, 'NaN': nan_count, '빈문자열/공백': blank_count})

missing_summary = pd.DataFrame(rows).set_index('column')
missing_summary['total_count'] = missing_summary['NaN'] + missing_summary['빈문자열/공백']
print(missing_summary.to_string())
print()
assert missing_summary['total_count'].sum() == 0, '결측이 발견되었습니다 — 처리 로직 추가 필요'
print('결측 0건 — 처리 대상 없음')

zero_all = (df[AMOUNT_COLS] == 0).all(axis=1)
print()
print(f"계 == 0            : {(df['계'] == 0).sum():,}건")
print(f"지출액 == 0        : {(df['지출액'] == 0).sum():,}건")
print(f"7개 금액 전부 0    : {zero_all.sum():,}건  (파생변수 is_zero_amount 로 표시 예정)")

step_log.append(('결측치 처리', df.shape[0], df.shape[1]))

        NaN  빈문자열/공백  total_count
column                           
지역        0        0            0
자치단체      0        0            0
회계        0        0            0
세부사업명     0        0            0
계         0        0            0
국비        0        0            0
시도비       0        0            0
시군구비      0        0            0
기타        0        0            0
지출액       0        0            0
집행잔액      0        0            0
분야        0        0            0
세부분야      0        0            0
연도        0        0            0

결측 0건 — 처리 대상 없음

계 == 0            : 527건
지출액 == 0        : 2,231건
7개 금액 전부 0    : 524건  (파생변수 is_zero_amount 로 표시 예정)


---
## 2. 이상치

세출 데이터에 IQR을 그대로 쓰면 안 된다. 예산액 분포는 본질적으로 극단적 우편향(광역 본청의 수백억 사업과
기초단체의 수백만원 사업이 한 컬럼에 섞임)이라, IQR을 돌리면 정상적인 대형 사업이 전부 이상치로 잡힌다.
"분포상 극단값"이 아니라 **"회계 규칙상 있을 수 없는 값"**만 이상치로 정의한다.

In [3]:
# 1) 금액 필드 음수값
print('[음수값] 컬럼별')
for col in AMOUNT_COLS:
    neg = int((df[col] < 0).sum())
    print(f'  {col:6s}: {neg}건')

# 2) IQR은 참고용으로만 계산 (채택하지 않음)
q1, q3 = df['지출액'].quantile(0.25), df['지출액'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print()
print(f'[참고] 지출액 IQR×1.5 상한 {upper:,.0f}원 초과 {(df["지출액"] > upper).sum():,}건 '
      f'({(df["지출액"] > upper).mean() * 100:.2f}%) → 정상적인 대형 사업이 대량 포함되므로 채택하지 않음')

# 3) 집행잔액 음수 == 지출액 > 계 인지 교차검증
neg_balance = df['집행잔액'] < 0
over_exec = df['지출액'] > df['계']
assert neg_balance.equals(over_exec), '집행잔액<0 과 지출액>계 가 일치하지 않음'
print()
print(f'집행잔액 < 0 (= 지출액 > 계) : {neg_balance.sum()}건 — 삭제하지 않고 플래그만 부여 (다음 단계)')

step_log.append(('이상치 처리', df.shape[0], df.shape[1]))

[음수값] 컬럼별
  계     : 0건
  국비    : 0건
  시도비   : 0건
  시군구비  : 0건
  기타    : 0건
  지출액   : 0건
  집행잔액  : 10건

[참고] 지출액 IQR×1.5 상한 659,762,745원 초과 6,155건 (12.66%) → 정상적인 대형 사업이 대량 포함되므로 채택하지 않음

집행잔액 < 0 (= 지출액 > 계) : 10건 — 삭제하지 않고 플래그만 부여 (다음 단계)


---
## 3. 손상 레코드

1. 회계 항등식 위반 — `계` = `국비+시도비+시군구비+기타`, `집행잔액` = `계−지출액`
2. 완전 중복행
3. 텍스트 손상 — HTML 엔티티, 공백
4. 동일 키 중복(금액 다름)

**채택 근거 (대화에서 실측 검증 완료):** 정규화(HTML 엔티티 디코딩→공백 정리)를 먼저 하고 중복 제거를 나중에 해야,
공백 한 칸 차이로 위장된 인천 옹진군 "홍보책자 제작" 중복(960만원)까지 잡힌다. 순서를 반대로 하면 이 1건을 놓친다.

In [4]:
# 3-1. 회계 항등식 / 완전중복(정규화 전) / 텍스트손상 / 동일키중복 — 현황 파악
sum_parts = df[['국비', '시도비', '시군구비', '기타']].sum(axis=1)
count_sum_mismatch = int((df['계'] != sum_parts).sum())
count_balance_mismatch = int((df['집행잔액'] != (df['계'] - df['지출액'])).sum())
count_full_duplicate_before = int(df.duplicated().sum())

count_html_entity = int(raw[TEXT_COLS].apply(
    lambda s: s.str.contains(r'&[a-zA-Z]+;|&#\d+;', regex=True)).any(axis=1).sum())
count_space_issue = int(raw[TEXT_COLS].apply(
    lambda s: (s != s.str.strip()) | s.str.contains(r'\s{2,}', regex=True)).any(axis=1).sum())

KEY_COLS = ['지역', '자치단체', '회계', '세부사업명', '연도']
count_key_duplicate = int(df.duplicated(subset=KEY_COLS).sum() - count_full_duplicate_before)

corrupted_summary = pd.DataFrame([
    {'유형': '계 != 국비+시도비+시군구비+기타', '건수': count_sum_mismatch},
    {'유형': '집행잔액 != 계-지출액', '건수': count_balance_mismatch},
    {'유형': '완전 중복행(정규화 전)', '건수': count_full_duplicate_before},
    {'유형': 'HTML 엔티티 미디코딩', '건수': count_html_entity},
    {'유형': '앞뒤/중복 공백', '건수': count_space_issue},
    {'유형': '동일 키 중복(금액 다름)', '건수': count_key_duplicate},
]).set_index('유형')
print(corrupted_summary.to_string())

assert count_sum_mismatch == 0, '회계 항등식 위반 발견 — 확인 필요'
assert count_balance_mismatch == 0, '잔액 항등식 위반 발견 — 확인 필요'

                      건수
유형                      
계 != 국비+시도비+시군구비+기타    0
집행잔액 != 계-지출액          0
완전 중복행(정규화 전)          1
HTML 엔티티 미디코딩         44
앞뒤/중복 공백             220
동일 키 중복(금액 다름)       310


In [5]:
# 3-2. 텍스트 정규화(HTML 엔티티 디코딩 -> 앞뒤 공백 제거 -> 중복 공백 1칸으로 축소)
def normalize_text(series: pd.Series) -> pd.Series:
    return (series.map(html.unescape)
                  .str.replace(r'\s+', ' ', regex=True)
                  .str.strip())

for col in TEXT_COLS:
    df[col] = normalize_text(df[col])

# 3-3. 완전 중복행 제거 (정규화 이후에 수행해야 정규화로 새로 드러나는 중복까지 잡힘)
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
removed = before - len(df)
print(f'완전 중복행 제거: {before:,}행 -> {len(df):,}행 ({removed}건 제거)')
assert removed == 2, f'예상(2건)과 다른 제거 건수: {removed}건 — 원본 데이터가 바뀌었는지 확인 필요'

# 3-4. 동일 키 중복(금액 다름) / 7개 금액 전부 0 : 정상 레코드로 판단해 유지 (재원별·차수별 분리 편성)
step_log.append(('손상 레코드 처리', df.shape[0], df.shape[1]))
print(f'손상 레코드 처리 완료: {df.shape}')

완전 중복행 제거: 48,619행 -> 48,617행 (2건 제거)
손상 레코드 처리 완료: (48617, 14)


---
## 4. 파생변수

| 컬럼 | 정의 | 출처/근거 |
|---|---|---|
| `local_own_revenue` | 시도비 + 시군구비 (국비 제외) | 가설2 핵심 지표, 방법론 확정사항 |
| `is_overexecuted` | 지출액 > 계 | 진경 — 삭제 대신 플래그(민감도 분석용) |
| `is_zero_amount` | 7개 금액 컬럼 전부 0 | 진경 — '사업 건수' 지표 계산 시 포함/제외 선택 가능하도록 |
| `account_type` | 회계 문자열 분류 (일반회계/특별회계/기금/기타) | 분류 로직은 진경, 순서: 기금→특별회계→일반회계→기타 |
| `account_type_code` | 위 유형의 숫자 코드 | **승희/팀 방법론 문서 코드 체계 채택**: 1=일반회계, 2=특별회계, 3=기금, 4=기타 |
| `region_type` | 광역시 / 도 | 이번 통합에서 신규 추가 — 국민여행조사와 조인 후 층화 분석용 |


In [6]:
# local_own_revenue: 지자체 자체재원 예산현액 (국비 제외) - 가설2 핵심 지표
df['local_own_revenue'] = df['시도비'] + df['시군구비']

# is_overexecuted: 예산현액을 넘겨 집행한 건
df['is_overexecuted'] = df['지출액'] > df['계']

# is_zero_amount: 7개 금액 컬럼이 전부 0인 건
df['is_zero_amount'] = (df[AMOUNT_COLS] == 0).all(axis=1)

# account_type: 진경의 분류 로직(단어 우선순위) 그대로 사용
def classify_account(value: str) -> str:
    if '기금' in value:
        return '기금'
    if '특별회계' in value:
        return '특별회계'
    if '일반회계' in value:
        return '일반회계'
    return '기타'

df['account_type'] = df['회계'].map(classify_account)

# account_type_code: 승희/팀 방법론 문서 코드 체계 채택 (1=일반/2=특별/3=기금/4=기타)
ACCOUNT_TYPE_CODE = {'일반회계': 1, '특별회계': 2, '기금': 3, '기타': 4}
df['account_type_code'] = df['account_type'].map(ACCOUNT_TYPE_CODE)

# region_type: 광역시(8) / 도(9) — 국민여행조사 유저가이드 141쪽 정의 기준 층화용
METRO = ['서울', '부산', '대구', '인천', '광주', '대전', '울산', '세종']
df['region_type'] = df['지역'].apply(lambda x: '광역시' if x in METRO else '도')

DERIVED_COLS = ['local_own_revenue', 'is_overexecuted', 'is_zero_amount',
                'account_type', 'account_type_code', 'region_type']

# 검증
assert (df['local_own_revenue'] <= df['계']).all(), 'local_own_revenue 가 계를 초과함'
assert df['is_overexecuted'].equals(df['집행잔액'] < 0), 'is_overexecuted 와 집행잔액 음수가 불일치'
assert df['account_type'].isna().sum() == 0 and df['account_type_code'].isna().sum() == 0
assert df['region_type'].isna().sum() == 0

step_log.append(('파생 변수 생성', df.shape[0], df.shape[1]))
print(df[DERIVED_COLS].head().to_string())
print()
print('is_overexecuted :', df['is_overexecuted'].sum(), '건')
print('is_zero_amount  :', df['is_zero_amount'].sum(), '건')
print()
print(df.groupby(['account_type', 'account_type_code']).size().to_string())
print()
print(df['region_type'].value_counts().to_string())

   local_own_revenue  is_overexecuted  is_zero_amount account_type  account_type_code region_type
0         1792000000            False           False         일반회계                  1         광역시
1          100000000            False           False         일반회계                  1         광역시
2                  0            False           False         일반회계                  1         광역시
3                  0            False           False         일반회계                  1         광역시
4         2810000000            False           False         일반회계                  1         광역시

is_overexecuted : 10 건
is_zero_amount  : 524 건

account_type  account_type_code
기금            3                      169
기타            4                       68
일반회계          1                    48183
특별회계          2                      197

region_type
도      40906
광역시     7711


---
## 5. 컬럼명 영문화 (승희 방식 채택)

`분야`(전 행 '문화및관광' 단일값, 분산 0)는 제외하고, 나머지 원본 컬럼 + 파생변수 전부를 영문 snake_case로 변경한다.

In [7]:
UNUSED_COLS = ['분야']   # 전 행 '문화및관광' 단일값 -> 분산 0, 어떤 집계·검정에도 기여하지 않음
df = df.drop(columns=UNUSED_COLS)

col_rename = {
    '지역': 'region',
    '자치단체': 'municipality',
    '회계': 'account_name',
    '세부사업명': 'project_name',
    '계': 'budget_total',
    '국비': 'budget_national',
    '시도비': 'budget_sido',
    '시군구비': 'budget_sigungu',
    '기타': 'budget_other',
    '지출액': 'expenditure',
    '집행잔액': 'balance',
    '세부분야': 'subsector',
    '연도': 'year',
    # 파생변수는 이미 영문이므로 그대로 유지 (local_own_revenue, is_overexecuted,
    # is_zero_amount, account_type, account_type_code, region_type)
}

missing_map = set(df.columns) - set(col_rename.keys()) - set(DERIVED_COLS)
assert not missing_map, f'매핑 안 된 컬럼 발견: {missing_map}'

df = df.rename(columns=col_rename)
print(df.shape)
print(df.columns.tolist())

(48617, 19)
['region', 'municipality', 'account_name', 'project_name', 'budget_total', 'budget_national', 'budget_sido', 'budget_sigungu', 'budget_other', 'expenditure', 'balance', 'subsector', 'year', 'local_own_revenue', 'is_overexecuted', 'is_zero_amount', 'account_type', 'account_type_code', 'region_type']


---
## 6. 저장 및 검증

In [8]:
df = df.reset_index(drop=True)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

step_log.append(('최종(processed_regional.csv)', df.shape[0], df.shape[1]))

summary = pd.DataFrame(step_log, columns=['단계', '행 수', '컬럼 수'])
summary['행 증감'] = summary['행 수'].diff().fillna(0).astype(int)
summary['컬럼 증감'] = summary['컬럼 수'].diff().fillna(0).astype(int)
print(summary.to_string(index=False))
print()
print(f'{OUT_PATH} 저장 완료: {df.shape[0]:,}행 x {df.shape[1]}열')

                        단계   행 수  컬럼 수  행 증감  컬럼 증감
                        원본 48619    14     0      0
                    결측치 처리 48619    14     0      0
                    이상치 처리 48619    14     0      0
                 손상 레코드 처리 48617    14    -2      0
                  파생 변수 생성 48617    20     0      6
최종(processed_regional.csv) 48617    19     0     -1

processed_regional.csv 저장 완료: 48,617행 x 19열


### (참고) 시/도 × 연도 집계 — 국민여행조사와 조인할 때 쓰는 함수

파일로 저장하지 않는다. 행 단위가 세부사업에서 시도×연도로 바뀌므로 processed_regional.csv 와 같은 파일에 담을 수 없고,
집계 조건(회계유형·초과집행 포함 여부 등)을 바꿔가며 여러 번 돌려야 하는 분석 단계 연산이기 때문이다 (진경 방침 그대로 채택).

In [9]:
def aggregate_sido_year(frame, account_types=None, include_overexec=True):
    f = frame.copy()
    if account_types is not None:
        f = f[f['account_type'].isin(account_types)]
    if not include_overexec:
        f = f[~f['is_overexecuted']]
    return (f.groupby(['region', 'year', 'region_type'], as_index=False)
             .agg(budget_total=('budget_total', 'sum'),
                  local_own_revenue=('local_own_revenue', 'sum'),
                  expenditure=('expenditure', 'sum'),
                  project_count=('project_name', 'size')))

region_year = aggregate_sido_year(df)
print(region_year.shape, '(17개 시도 x 3개년 = 51행)')
print(region_year.head(10).to_string(index=False))

(51, 7) (17개 시도 x 3개년 = 51행)
region  year region_type  budget_total  local_own_revenue  expenditure  project_count
    강원  2023           도  930717885518       762866316697 600579015199           1693
    강원  2024           도  979666557001       801065663512 689219801520           1628
    강원  2025           도  956022582433       783158142302 683777504767           1617
    경기  2023           도  870909988014       664527585796 647880379832           1851
    경기  2024           도 1037085702464       827456262703 835762958225           1859
    경기  2025           도  879777186034       689372881651 695014452750           1872
    경남  2023           도  941138585226       744524998728 644658724658           1890
    경남  2024           도  989215940058       763696439328 668683687745           1944
    경남  2025           도 1064651320105       795809928525 769415509643           2059
    경북  2023           도 1361925312600      1045064013605 982146200351           2156
